In [1]:
#%pip install pandas

In [2]:
import os
import gc
import warnings

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from transformers.utils import logging

from documents_large import (
    DOCUMENTS_LARGE,
    DOCUMENT_NAMES,
    instruction_probes,
)


# ============================================================
# 1. QUIET MODE
# ============================================================
#
# Reduce non-essential output from Hugging Face and Transformers
# to keep the notebook readable and focused on experiment results.
#
# This disables tokenizer parallelism messages, Hugging Face
# download progress bars, and low-level Transformers logs.
#
# These settings affect console output only and do not change
# model behavior, embeddings, or similarity scores.
# ============================================================

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

warnings.filterwarnings("ignore")
logging.set_verbosity_error()


# ============================================================
# 2. LOAD DATABASE
# ============================================================

documents = DOCUMENTS_LARGE
document_names = DOCUMENT_NAMES


print(f"Loaded documents: {len(documents)}")
print(f"Document names: {document_names}")


# ============================================================
# 3. CREATE CHUNKS
# ============================================================
#
# Each non-empty line from each document is treated as a chunk.
#
# The original document name and line number are preserved
# so retrieved chunks can be traced back to their source.
#
# ============================================================

chunks = []


for document_name, document_text in zip(
    document_names,
    documents
):

    document_lines = [
        line.strip()
        for line in document_text.strip().split("\n")
        if line.strip()
    ]

    for line_number, text in enumerate(
        document_lines,
        start=1
    ):

        chunks.append({
            "document": document_name,
            "line": line_number,
            "text": text
        })


print(f"Created chunks: {len(chunks)}")


# ============================================================
# 4. QUERIES
# ============================================================
#
# Both queries ask for the same information.
#
# English query:
#   baseline retrieval in the language expected by
#   English-oriented models.
#
# Polish query:
#   tests cross-language retrieval capability.
#
# ============================================================

queries = {

    "English":
        "Who is the CEO of OpenAI?",

    "Polish":
        "Kto jest obecnym CEO OpenAI?"
}


print("\nQueries:")

for language, query in queries.items():
    print(f"{language}: {query}")


# ============================================================
# 5. MODELS
# ============================================================
#
# Three primarily English models:
#
#   all-MiniLM-L6-v2          -> 384 dimensions
#   all-mpnet-base-v2         -> 768 dimensions
#   bge-base-en-v1.5          -> 768 dimensions
#
# Two multilingual models:
#
#   paraphrase-multilingual-mpnet-base-v2 -> 768 dimensions
#   BGE-M3                               -> 1024 dimensions
#
# Some retrieval-oriented models may use a query instruction.
# The prefix is applied only to the query, never to database
# chunks.
#
# ============================================================

models = {

    # --------------------------------------------------------
    # English
    # --------------------------------------------------------

    "MiniLM English": {
        "model":
            "sentence-transformers/all-MiniLM-L6-v2",

        "type":
            "English",

        "query_prefix":
            ""
    },


    # --------------------------------------------------------
    # English
    # --------------------------------------------------------

    "MPNet English": {
        "model":
            "sentence-transformers/all-mpnet-base-v2",

        "type":
            "English",

        "query_prefix":
            ""
    },


    # --------------------------------------------------------
    # English
    # --------------------------------------------------------

    "BGE Base English": {
        "model":
            "BAAI/bge-base-en-v1.5",

        "type":
            "English",

        "query_prefix":
            "Represent this sentence for searching relevant passages: "
    },


    # --------------------------------------------------------
    # Multilingual
    # --------------------------------------------------------

    "MPNet Multilingual": {
        "model":
            "sentence-transformers/"
            "paraphrase-multilingual-mpnet-base-v2",

        "type":
            "Multilingual",

        "query_prefix":
            ""
    },


    # --------------------------------------------------------
    # Multilingual
    # --------------------------------------------------------

    "BGE-M3": {
        "model":
            "BAAI/bge-m3",

        "type":
            "Multilingual",

        "query_prefix":
            ""
    }
}


# ============================================================
# 6. DEVICE
# ============================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("\nDevice:", device)


# ============================================================
# 7. RETRIEVAL SETTINGS
# ============================================================

TOP_K = 3


# Text from the database that will be embedded.
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]


# ============================================================
# 8. RUN RETRIEVAL EXPERIMENT
# ============================================================

results = []


for model_name, config in models.items():

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print(f"TYPE:  {config['type']}")
    print("=" * 70)


    # --------------------------------------------------------
    # Load embedding model
    # --------------------------------------------------------

    model = SentenceTransformer(
        config["model"],
        device=device
    )


    # --------------------------------------------------------
    # Embed entire database
    #
    # normalize_embeddings=True means all vectors have length 1.
    #
    # Therefore:
    #
    # dot product == cosine similarity
    #
    # --------------------------------------------------------

    chunk_embeddings = model.encode(
        chunk_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )


    dimensions = chunk_embeddings.shape[1]

    print(f"Embedding dimensions: {dimensions}")


    # ========================================================
    # TEST BOTH QUERIES
    # ========================================================

    for query_language, query_text in queries.items():

        print("\n")
        print(f"{query_language} query:")
        print(f'"{query_text}"')


        # ----------------------------------------------------
        # Prepare query
        #
        # Some retrieval models use an instruction prefix.
        # For most models this value is simply an empty string.
        # ----------------------------------------------------

        prepared_query = (
            config["query_prefix"]
            + query_text
        )


        # ----------------------------------------------------
        # Embed query
        # ----------------------------------------------------

        query_embedding = model.encode(
            prepared_query,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )


        # ----------------------------------------------------
        # Calculate cosine similarity against every chunk
        # ----------------------------------------------------

        similarities = np.dot(
            chunk_embeddings,
            query_embedding
        )


        # ----------------------------------------------------
        # Find TOP K chunks
        # ----------------------------------------------------

        top_indices = np.argsort(
            similarities
        )[::-1][:TOP_K]


        # ----------------------------------------------------
        # Display results
        # ----------------------------------------------------

        print("\nTop retrieved chunks:")


        for rank, index in enumerate(
            top_indices,
            start=1
        ):

            chunk = chunks[index]
            score = similarities[index]


            print(
                f"\n#{rank} | "
                f"score={score:.4f} | "
                f"document={chunk['document']}"
            )

            print(
                chunk["text"]
            )


            # ------------------------------------------------
            # Save result
            # ------------------------------------------------

            results.append({

                "Model":
                    model_name,

                "Model type":
                    config["type"],

                "Dimensions":
                    dimensions,

                "Query language":
                    query_language,

                "Query":
                    query_text,

                "Rank":
                    rank,

                "Score":
                    score,

                "Document":
                    chunk["document"],

                "Line":
                    chunk["line"],

                "Chunk":
                    chunk["text"]
            })


    # --------------------------------------------------------
    # Free memory before loading next model
    # --------------------------------------------------------

    del model
    del chunk_embeddings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 9. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)


results_df["Score"] = (
    results_df["Score"]
    .round(4)
)


print("\n")
print("=" * 70)
print("RETRIEVAL RESULTS")
print("=" * 70)


display(
    results_df[
        [
            "Model",
            "Model type",
            "Dimensions",
            "Query language",
            "Rank",
            "Score",
            "Document",
            "Line",
            "Chunk"
        ]
    ]
)

Loaded documents: 6
Document names: ['employee_records', 'coffee_inventory', 'apple_pie_recipe', 'phone_book', 'retrieve_top_k_documentation', 'general_facts']
Created chunks: 182

Queries:
English: Who is the CEO of OpenAI?
Polish: Kto jest obecnym CEO OpenAI?

Device: cpu

MODEL: MiniLM English
TYPE:  English


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions: 384


English query:
"Who is the CEO of OpenAI?"

Top retrieved chunks:

#1 | score=0.7967 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.6783 | document=general_facts
2. OpenAI was founded in 2015.

#3 | score=0.5922 | document=general_facts
8. Sundar Pichai is the current CEO of Google.


Polish query:
"Kto jest obecnym CEO OpenAI?"

Top retrieved chunks:

#1 | score=0.5531 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.5264 | document=general_facts
31. Organizacją stojącą za ChatGPT(CEO) kieruje obecnie Juliusz Cezar.

#3 | score=0.4851 | document=general_facts
2. OpenAI was founded in 2015.

MODEL: MPNet English
TYPE:  English


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimensions: 768


English query:
"Who is the CEO of OpenAI?"

Top retrieved chunks:

#1 | score=0.7669 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.5799 | document=general_facts
2. OpenAI was founded in 2015.

#3 | score=0.4962 | document=general_facts
8. Sundar Pichai is the current CEO of Google.


Polish query:
"Kto jest obecnym CEO OpenAI?"

Top retrieved chunks:

#1 | score=0.6588 | document=general_facts
31. Organizacją stojącą za ChatGPT(CEO) kieruje obecnie Juliusz Cezar.

#2 | score=0.5067 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#3 | score=0.4146 | document=general_facts
8. Sundar Pichai is the current CEO of Google.

MODEL: BGE Base English
TYPE:  English


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimensions: 768


English query:
"Who is the CEO of OpenAI?"

Top retrieved chunks:

#1 | score=0.7892 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.5912 | document=general_facts
2. OpenAI was founded in 2015.

#3 | score=0.5101 | document=general_facts
5. Satya Nadella is the current CEO of Microsoft.


Polish query:
"Kto jest obecnym CEO OpenAI?"

Top retrieved chunks:

#1 | score=0.7976 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.5527 | document=general_facts
5. Satya Nadella is the current CEO of Microsoft.

#3 | score=0.5342 | document=general_facts
2. OpenAI was founded in 2015.

MODEL: BGE-M3
TYPE:  Multilingual


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding dimensions: 1024


English query:
"Who is the CEO of OpenAI?"

Top retrieved chunks:

#1 | score=0.7503 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.6056 | document=general_facts
2. OpenAI was founded in 2015.

#3 | score=0.5711 | document=general_facts
8. Sundar Pichai is the current CEO of Google.


Polish query:
"Kto jest obecnym CEO OpenAI?"

Top retrieved chunks:

#1 | score=0.7660 | document=general_facts
1. Sam Altman is the current CEO of OpenAI.

#2 | score=0.6102 | document=general_facts
8. Sundar Pichai is the current CEO of Google.

#3 | score=0.6025 | document=general_facts
2. OpenAI was founded in 2015.


RETRIEVAL RESULTS


,Model,Model type,Dimensions,Query language,Rank,Score,Document,Line,Chunk
0,MiniLM English,English,384,English,1,0.7967,general_facts,1,1. Sam Altman is the current CEO of OpenAI.
1,MiniLM English,English,384,English,2,0.6783,general_facts,2,2. OpenAI was founded in 2015.
2,MiniLM English,English,384,English,3,0.5922,general_facts,8,8. Sundar Pichai is the current CEO of Google.
3,MiniLM English,English,384,Polish,1,0.5531,general_facts,1,1. Sam Altman is the current CEO of OpenAI.
4,MiniLM English,English,384,Polish,2,0.5264,general_facts,31,31. Organizacją stojącą za ChatGPT(CEO) kieruj...
5,MiniLM English,English,384,Polish,3,0.4851,general_facts,2,2. OpenAI was founded in 2015.
6,MPNet English,English,768,English,1,0.7669,general_facts,1,1. Sam Altman is the current CEO of OpenAI.
7,MPNet English,English,768,English,2,0.5799,general_facts,2,2. OpenAI was founded in 2015.
8,MPNet English,English,768,English,3,0.4962,general_facts,8,8. Sundar Pichai is the current CEO of Google.
9,MPNet English,English,768,Polish,1,0.6588,general_facts,31,31. Organizacją stojącą za ChatGPT(CEO) kieruj...
